# Notebook 5.1  Alignments, blanks, and beams: end-to-end ASR by hand

**Companion to Chapter 5, *Introduction to Arabic Speech Technologies*.**

**Goal.** Make the three end-to-end ideas concrete on tiny examples, with no training and no
downloads: CTC alignments and the CTC sum, a monotonic RNN-Transducer walk, a beam search with
shallow language-model fusion, and a tokenization comparison. An Exercise solutions section follows.

## 1. CTC: enumerate and collapse alignments

In [ ]:
from itertools import product
def collapse(seq):
    out=[]
    for s in seq:
        if not out or out[-1]!=s: out.append(s)   # merge repeats
    return [s for s in out if s!='-']              # drop blanks

def valid_alignments(target, n_frames, symbols):
    return [''.join(p) for p in product(symbols, repeat=n_frames) if collapse(p)==list(target)]

al = valid_alignments('AB', 4, ['A','B','-'])
print('number of length-4 alignments collapsing to A B:', len(al))
print(al)

## 2. The CTC sum (toy network output)

CTC scores a transcript as the total probability of every alignment that collapses to it. With a
toy per-frame probability table we add those alignment probabilities.

In [ ]:
import numpy as np
syms=['A','B','-']; idx={s:i for i,s in enumerate(syms)}
# per-frame probabilities over {A,B,blank} for 4 frames (rows sum to 1)
Pf=np.array([[0.6,0.1,0.3],[0.5,0.2,0.3],[0.1,0.6,0.3],[0.1,0.7,0.2]])
def path_prob(p): return float(np.prod([Pf[t, idx[s]] for t,s in enumerate(p)]))
total=sum(path_prob(p) for p in al)
print('P(target = A B) summed over all valid alignments = {:.4f}'.format(total))
best=max(al, key=path_prob)
print('most probable single alignment:', best, '(prob {:.4f})'.format(path_prob(best)))

## 3. RNN-Transducer: a monotonic emit-or-advance walk

An RNN-T moves through the audio monotonically: at each step it either emits a token or advances
one frame. Here we follow a fixed policy to show the monotonic path; a real model learns the choice.

In [ ]:
def rnnt_walk(emissions):
    # emissions: list over frames of the tokens to emit at that frame (already decided)
    t=0; out=[]; trace=[]
    for frame_tokens in emissions:
        for tok in frame_tokens:
            out.append(tok); trace.append(f'emit {tok} at frame {t}')
        trace.append(f'advance to frame {t+1}'); t+=1
    return out, trace
out,trace=rnnt_walk([['h'],[],['i'],[]])   # emit h, advance, emit i, advance
print('output:', ''.join(out))
for line in trace: print(' ', line)
print('the time index never decreases: the alignment is monotonic.')

## 4. Beam search with shallow language-model fusion

Shallow fusion adds a weighted language-model score to the acoustic score during the beam search.
We decode a tiny 2-step example and watch the language model change the winner.

In [ ]:
# acoustic scores (log-prob) for each candidate token at two steps
ac={'0':{'shu':-0.7,'maa':-0.8}, '1':{'nu':-0.3,'dha':-0.4}}
# a toy language model: prefers the formal 'maa dha' over the dialectal 'shu nu'
lm={('shu','nu'):-2.0, ('maa','dha'):-0.5, ('shu','dha'):-3.0, ('maa','nu'):-3.0}
import itertools
def decode(lm_weight):
    best=None
    for a,b in itertools.product(ac['0'], ac['1']):
        s=ac['0'][a]+ac['1'][b]+lm_weight*lm[(a,b)]
        if best is None or s>best[0]: best=(s,(a,b))
    return best[1]
print('lambda=0.0 (audio only):', decode(0.0))
print('lambda=1.0 (with LM)  :', decode(1.0))
print('=> with enough LM weight the fluent formal hypothesis wins; too much LM overrides the audio.')

## 5. Output units: tokenization comparison

In [ ]:
sentence_words=['سيكتبونها','للمكتبة']
chars=[ch for w in sentence_words for ch in w]
bpe=['sy','ktbwn','ha','ll','mktba']
morph=['sa-','yaktubuun','-ha','li-','al-','maktaba']
for name,toks in [('words',sentence_words),('characters',chars),('BPE subwords',bpe),('morphemes',morph)]:
    print(f'{name:14} count={len(toks):2}  {toks}')

## 6. Exercise solutions

Exercise 1 is solved by the code in Section 1 (fifteen valid alignments); the rest are summarized.

**Exercise 1.** The function in Section 1 enumerates all length-4 strings over {A, B, blank} and keeps the fifteen that collapse to 'A B'. An invalid example: 'AAAA' collapses to 'A' (missing B).

In [ ]:
inval='AAAA'; print(inval, '-> collapses to', ''.join(collapse(inval)), '(invalid: missing B)')
inval2='B-A-'; print(inval2, '-> collapses to', ''.join(collapse(inval2)), '(invalid: wrong order)')

**Exercise 2 (design).** RNN-T is monotonic because at each step it only emits or advances and never moves back in time, so it can emit with bounded delay as audio arrives (streaming). An attention decoder may attend to any frame, including future ones, so it can need the whole utterance first.

**Exercise 3.** Shallow fusion adds lambda times the language-model score to each candidate during the beam search (Section 4). If lambda is too high the language model dominates and the decoder produces fluent text that no longer matches the audio.

**Exercise 4 (design).** Streaming voice assistant: RNN-Transducer (or streaming CTC), for low latency. Offline broadcast transcription: hybrid CTC/attention, for highest accuracy with robust decoding.

**Exercise 5 (design).** For Modern Standard Arabic broadcast, subwords (BPE/SentencePiece) balance length and vocabulary; for unstandardized dialect, subwords or characters absorb spelling variation and avoid out-of-vocabulary forms (Section 5).